In [ ]:
# imports

import os
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr

In [ ]:
# Load environment variables in a file called .env
# Print the key prefixes to help with any debugging

load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")

In [ ]:
# Initialize

openai = OpenAI()
MODEL = 'gpt-4.1-mini'

In [ ]:
system_message = """You are an expert assistant at INVENSITY, an international technology 
and innovation consultancy. 
Your goal is to identify technical challenges within the customer's projects. 
You should subtly inquire about their current situation in areas such as Systems Engineering, 
Software Development, or Safety Management, building up the clients' trust until they reveal their pain points
so that you can then ask 'And, are you satisfied with how things are now?'.

Mention our competitive advantages: a '95% customer return rate' and our '9 Centers of Excellence.' if the 
client asks questions about Invensity. 

If the customer expresses uncertainty regarding their development process, encourage them to explore 
our 'Management Consulting' or 'Software & Data Sciences' services. For example, if the customer says 
'We are having issues with software security,' you could reply: 'I understand—cybersecurity is critical 
in today's landscape. At INVENSITY, we have experts in our Centers of Excellence who specialize 
specifically in strengthening software architectures and risk management.'"""

In [ ]:

def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    relevant_system_message = system_message
    if 'belt' in message.lower():
        relevant_system_message += " The store does not sell belts; if you are asked for belts, be sure to point out other items on sale."
    
    messages = [{"role": "system", "content": relevant_system_message}] + history + [{"role": "user", "content": message}]

    stream = openai.chat.completions.create(model=MODEL, messages=messages, stream=True)

    response = ""
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        yield response

In [ ]:
gr.ChatInterface(fn=chat, type="messages").launch()